In [ ]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup

import pandas as pd

from pandas import ExcelWriter

from time import sleep

import datetime

from selenium import webdriver

from selenium.webdriver.common.by import By

import os

# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'HK HKMA' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename= '{} data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

#scriptfolder = f"C:\\Users\\siewekoa\\OneDrive - moodys.com\\Desktop\\My_data\\Project_work\\scripts_regulator\\{regulatorName}" ## to comment for the production environment

scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert

         }

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = {

            "HK HKMA 1": "List of licensed banks",

            "HK HKMA 2": "List of restricted licence banks", 

		    "HK HKMA 3": "List of deposit-taking companies", 

            "HK HKMA 4": "List of local representative offices",

            }



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}



location = ['Incorporated in Hong Kong', 'Incorporated outside Hong Kong', 'Offices in Hong Kong']

processdate = now.strftime('%Y-%m-%d')



# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : - {fileType} file = {os.listdir(tempfolder)})")

            break

        else:

            print(f"[INFO] : - Download {fileType} file ... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : - Failed to Download {fileType} file. Run Script again' )

    return  os.listdir(tempfolder)[0]



# %%

#------------------------------------------------ Begin_Main ----------------------------------------

driver.get("https://www.hkma.gov.hk/eng/key-functions/banking/banking-regulatory-and-supervisory-regime/the-three-tier-banking-system/")



for k, reg in enumerate(regdict):

	activation = False

	cut_start = 2



	print(f"[INFO] : Working {k+1}/{len(regdict)} | {reg} | {regdict[reg]}")

	driver.find_element(By.XPATH, f"//a[contains(text(),'{regdict[reg]}')]").click()

	file =  check_dowload_files(tempfolder, "xls")

	filePath = os.path.join(tempfolder, file)

	df = pd.read_excel(filePath)

	df.columns = ['Name']



	if reg == 'HK HKMA 4' :

		new_row = {'Name': 'Offices in Hong Kong'}

		df.loc[1] = new_row

		cut_start = 1

		

	df = df.fillna("")  # subsitute nan with empty strings

	df = df[cut_start:]

	df = df.reset_index(drop=True)



	print(f"[INFO] : - DataFrame '{file}' | containe = {df.shape}")

	for index, row in df.iterrows():



		if row['Name'].strip() in location :

			activation = True

			continue

		elif len(row['Name'].strip()) == 0:

			activation = False

		

		if activation :

			sqldict['Name'].append(row['Name'].strip())

			sqldict["Cntry"].append("HK") 

			sqldict['ListProcessDate'].append(processdate)

			sqldict['RegCtry'].append(reg.split()[0])

			sqldict['RegCode'].append(reg.split()[1])

			sqldict['ListCode'].append(reg.split()[2])



	sleep(1)

	sqldict = bourange_same_length_array(sqldict)

	for del_file in os.listdir(tempfolder):

		os.remove(os.path.join(tempfolder, del_file))



# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
    
    